# UR10 Evaluation Comparison
Pulls all eval runs from W&B and creates a **policy × condition** histogram grid.

Fully adaptive: add more sweeps to W&B and re-run this notebook.

In [ ]:
import wandb
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, OrderedDict
import os

# ── Config ──
ENTITY = "weissma6-zhaw-school-of-engineering"
PROJECT = "UR10_pick_ppo"
EVAL_GROUP = "eval"  # group tag used in run_one_eval.py

# Condition display order (add more as needed)
CONDITION_ORDER = ["Default", "Mass", "Friction", "Mass+Friction"]

# Output
os.makedirs("evaluation/graphs", exist_ok=True)
SAVE_PATH = "evaluation/graphs/eval_comparison_grid"

## 1. Pull eval runs from W&B

In [ ]:
api = wandb.Api()
runs = api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={"group": EVAL_GROUP},
)
print(f"Found {len(runs)} eval runs\n")

# ── Parse into grid ──
# results[policy_label][condition] = np.array of rewards
results = defaultdict(dict)
meta = []  # for the summary table

for run in runs:
    cfg = run.config
    policy_run_id = cfg.get("policy_run_id", "unknown")
    condition = cfg.get("condition", "Default")
    eval_id = cfg.get("eval_id", run.name)
    
    # Shorten policy label: strip timestamp suffix
    policy_label = policy_run_id.split("_20")[0] if "_20" in policy_run_id else policy_run_id
    
    # Try to get rewards from summary or artifact
    rewards = None
    
    # Method 1: rewards_table logged as wandb.Table
    try:
        arts = [a for a in run.logged_artifacts() if a.type == "eval"]
        if arts:
            art_dir = arts[0].download(root="_eval_artifacts")
            import json
            for fname in os.listdir(art_dir):
                if fname.endswith(".json"):
                    with open(os.path.join(art_dir, fname)) as f:
                        data = json.load(f)
                    if "rewards" in data:
                        rewards = np.array(data["rewards"])
    except Exception as e:
        print(f"  ⚠ Could not load artifact for {eval_id}: {e}")
    
    # Method 2: fallback to summary stats (no histogram possible)
    if rewards is None:
        mean = run.summary.get("mean_reward")
        std = run.summary.get("std_reward")
        if mean is not None:
            print(f"  ⚠ {eval_id}: only summary stats (no raw rewards)")
            rewards = np.array([mean])  # placeholder
    
    if rewards is not None:
        results[policy_label][condition] = rewards
        meta.append({
            "eval_id": eval_id,
            "policy_label": policy_label,
            "condition": condition,
            "mean": float(rewards.mean()),
            "std": float(rewards.std()),
            "n": len(rewards),
        })
        print(f"  ✓ {policy_label} / {condition}: mean={rewards.mean():.1f} ± {rewards.std():.1f} (n={len(rewards)})")
    else:
        print(f"  ✗ {eval_id}: no rewards found")

print(f"\n✓ Loaded {len(meta)} eval cells")

## 2. Summary Table

In [ ]:
# ── Sort policies and conditions ──
policy_labels = list(results.keys())

# Sort conditions by predefined order, unknowns at the end
all_conditions = set()
for p in results.values():
    all_conditions.update(p.keys())
cond_labels = sorted(
    all_conditions,
    key=lambda c: CONDITION_ORDER.index(c) if c in CONDITION_ORDER else 99,
)

print(f"Policies ({len(policy_labels)}): {policy_labels}")
print(f"Conditions ({len(cond_labels)}): {cond_labels}")

# ── Print table ──
header = f"{'Policy':<40}"
for c in cond_labels:
    header += f" {c:>20}"
print("\n" + header)
print("─" * len(header))

for policy in policy_labels:
    row = f"{policy:<40}"
    for cond in cond_labels:
        if cond in results[policy]:
            rews = results[policy][cond]
            row += f" {rews.mean():>8.1f} ± {rews.std():<8.1f}"
        else:
            row += f" {'—':>20}"
    print(row)

## 3. Histogram Grid Plot

In [ ]:
n_rows = len(policy_labels)
n_cols = len(cond_labels)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False,
)

# Global axis range for comparability
all_rewards = [
    results[p][c]
    for p in policy_labels for c in cond_labels
    if c in results[p] and len(results[p][c]) > 1
]
if all_rewards:
    global_min = min(r.min() for r in all_rewards)
    global_max = max(r.max() for r in all_rewards)
    pad = (global_max - global_min) * 0.1
    hist_bins = np.linspace(global_min - pad, global_max + pad, 20)
else:
    hist_bins = 15

for row, policy in enumerate(policy_labels):
    for col, cond in enumerate(cond_labels):
        ax = axes[row][col]

        if cond not in results[policy] or len(results[policy][cond]) <= 1:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=12, color="gray")
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            rews = results[policy][cond]
            ax.hist(rews, bins=hist_bins, color="#2E86AB", edgecolor="white", alpha=0.8)
            ax.axvline(rews.mean(), color="red", linestyle="--", linewidth=2,
                       label=f"μ={rews.mean():.1f}")
            ax.axvline(rews.mean() - rews.std(), color="gray", linestyle=":", linewidth=1.2)
            ax.axvline(rews.mean() + rews.std(), color="gray", linestyle=":", linewidth=1.2,
                       label=f"σ={rews.std():.1f}")
            ax.legend(fontsize=8, loc="upper right")

        # Labels
        if row == 0:
            ax.set_title(cond, fontsize=13, fontweight="bold")
        if col == 0:
            ax.set_ylabel(policy, fontsize=10, fontweight="bold")
        if row == n_rows - 1:
            ax.set_xlabel("Episode Reward", fontsize=10)

fig.suptitle(
    f"Robustness Comparison ({len(meta)} evals)",
    fontsize=15, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{SAVE_PATH}.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"\n✓ Saved to {SAVE_PATH}.pdf / .png")

## 4. Box Plot Comparison (compact view)

In [ ]:
fig, axes = plt.subplots(
    1, n_cols,
    figsize=(5 * n_cols, 5),
    sharey=True, squeeze=False,
)

colors = plt.cm.Set2(np.linspace(0, 1, n_rows))

for col, cond in enumerate(cond_labels):
    ax = axes[0][col]
    box_data = []
    box_labels = []
    box_colors = []

    for row, policy in enumerate(policy_labels):
        if cond in results[policy] and len(results[policy][cond]) > 1:
            box_data.append(results[policy][cond])
            box_labels.append(policy)
            box_colors.append(colors[row])

    if box_data:
        bp = ax.boxplot(
            box_data, labels=box_labels, patch_artist=True,
            widths=0.6, showmeans=True,
            meanprops=dict(marker="D", markerfacecolor="red", markersize=6),
        )
        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.tick_params(axis="x", rotation=30)

    ax.set_title(cond, fontsize=13, fontweight="bold")
    if col == 0:
        ax.set_ylabel("Episode Reward", fontsize=12)
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Robustness Comparison (Box Plots)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}_boxplot.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{SAVE_PATH}_boxplot.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"✓ Saved box plot")